# Floyd–Warshall: All-Pairs Shortest Paths

## Introduction
The **Floyd–Warshall algorithm** finds the shortest distance between **every pair of vertices** in a weighted graph in a single run. This differs from Dijkstra's or Bellman–Ford, which compute shortest paths from one source at a time.

### Core idea (dynamic programming)
Let `d_k[i][j]` be the shortest distance from vertex `i` to vertex `j` using only vertices `{0, 1, ..., k-1}` as intermediate stops. At each step we ask: *is it cheaper to go through vertex `k`?*

d_{k+1}[i][j] = min( d_k[i][j],  d_k[i][k] + d_k[k][j] )

After all `n` vertices have been considered as intermediates, `d_n[i][j]` is the true shortest distance from `i` to `j`.

### What this notebook does
1. Imports NumPy for matrix handling.
2. Implements Floyd–Warshall with a "previous / current" matrix pair.
3. Builds an 8-vertex directed, weighted graph as an adjacency matrix.
4. Runs the algorithm and prints the shortest-path matrix.
5. Interprets the results and analyses time and space complexity.

## Step 1: Import Libraries
We import **NumPy**, which provides the 2-D array (`ndarray`) used to store the graph as an adjacency matrix. It also gives us the `.copy()` method for duplicating the matrix at each stage of the algorithm.

In [20]:
import numpy as np

## Step 2: Implement the Floyd–Warshall Function

`Floyd_Warshall(graph)` takes an `n x n` adjacency matrix, where `graph[i][j]` is the weight of the edge from `i` to `j`, and returns the matrix of shortest distances.

### How the code works
| Line | Purpose |
|---|---|
| `n = len(graph)` | Number of vertices. |
| `prev = graph` | Holds the distance matrix from the previous stage (initially the input graph). |
| `for x in range(n)` | **Outer loop:** picks `x` as the intermediate vertex being considered. |
| `curr = prev.copy()` | Creates a fresh matrix for this stage, so updates never read partially updated values. |
| `for i`, `for j` | Visit every ordered pair of vertices `(i, j)`. |
| `curr[i][j] = min(prev[i][j], prev[i][x] + prev[x][j])` | **Relaxation step:** keep the current best, or reroute through `x` if that is shorter. |
| `prev = curr` | The updated matrix becomes the baseline for the next intermediate vertex. |
| `return curr` | After all `n` intermediates are processed, `curr` holds the final answers. |

### Design notes
- The order of loops matters: the **intermediate vertex `x` must be the outermost loop**, which this code does correctly.
- Because each stage reads only from `prev` and writes only to `curr`, the input `graph` is **not modified** (`prev.copy()` creates a new array before any write).
- Classic implementations update a single matrix in place, which is also correct and saves memory. This version trades a little memory for clarity.

In [21]:
def Floyd_Warshall(graph):
    n = len(graph)
    prev = graph
    for x in range(n): 
        curr = prev.copy()
        for  i in range(n):
            for j in range(n):
                curr[i][j] = min(prev[i][j] , prev[i][x] + prev[x][j])
        prev = curr


    return curr



## Step 3: Define the Input Graph

The graph is a **complete, directed, weighted graph with 8 vertices (0–7)**, stored as an `8 x 8` NumPy integer matrix.

- `graph[i][j]` = cost of the direct edge from vertex `i` to vertex `j`.
- The **diagonal is 0**, since the distance from a vertex to itself is zero.
- The matrix is **not symmetric** (for example, `graph[0][1] = 12` but `graph[1][0] = 8`), so edges are directed and the cost depends on direction.
- Every pair has a finite direct edge, so no `INF` placeholders are needed. For sparse graphs, missing edges would normally be represented by a large value such as `np.inf`.
- All weights are positive, so there are no negative cycles to worry about.

Even though every direct edge exists, the direct edge is often **not** the cheapest route, and that is what the algorithm will discover.

In [22]:
graph = np.array([
    [0,  12,  7,  20,  15,  9,  18,  25],
    [8,   0, 14,   6,  19, 11,  22,  13],
    [5,  16,  0,  10,   4, 17,  12,  21],
    [14,  3,  9,   0,  13,  7,  16,   5],
    [11, 18,  6,  15,   0,  8,   3,  17],
    [7,  10, 19,   4,  12,  0,  14,   6],
    [16,  5, 11,  18,   7, 13,   0,   9],
    [9,  14,  3,  11,  16,  5,   8,   0]
], dtype=int)

## Step 4: Run the Algorithm and Display the Result

We call `Floyd_Warshall(graph)` and print the returned matrix using an f-string. The output is an `8 x 8` matrix where entry `[i][j]` is the **length of the shortest possible path from vertex `i` to vertex `j`**, possibly passing through several intermediate vertices.

In [23]:
print(f"Shortest path for each pair:\n{Floyd_Warshall(graph)}")

Shortest path for each pair:
[[ 0 12  7 13 11  9 14 15]
 [ 8  0 14  6 18 11 19 11]
 [ 5 12  0 10  4 12  7 15]
 [11  3  8  0 12  7 13  5]
 [11  8  6 12  0  8  3 12]
 [ 7  7  9  4 12  0 14  6]
 [13  5 11 11  7 13  0  9]
 [ 8 12  3  9  7  5  8  0]]


## Results and Interpretation

The printed matrix is the final all-pairs shortest-path table. Comparing it with the original graph shows where indirect routes beat direct edges.

| Pair | Direct edge | Shortest path | Route | Explanation |
|---|---|---|---|---|
| 0 → 3 | 20 | **13** | 0 → 5 → 3 | 9 + 4 = 13 |
| 3 → 0 | 14 | **11** | 3 → 1 → 0 | 3 + 8 = 11 |
| 1 → 6 | 22 | **19** | 1 → 3 → 7 → 6 | 6 + 5 + 8 = 19 |

### Observations
- **Every diagonal entry is 0**, as expected.
- Many entries are smaller than the corresponding direct edges, showing that indirect routes are often cheaper.
- The result matrix is **still asymmetric**, because the graph is directed (for example, 0 → 3 costs 13 but 3 → 0 costs 11).
- No entry in the result is larger than its direct edge, which confirms the algorithm only ever improves distances.
- The notebook outputs distances only. To recover the actual routes, you would additionally maintain a `next[i][j]` (or predecessor) matrix updated whenever a shorter path is found.

## Complexity Analysis

Let **n** be the number of vertices (here n = 8).

### Time complexity: O(n³)
- The outer loop over the intermediate vertex `x` runs `n` times.
- For each `x`, the two inner loops visit all `n x n` pairs `(i, j)`, doing constant work per pair (one addition and one `min`).
- Total relaxations: `n x n x n = n³` (512 for this graph).
- The `prev.copy()` at each stage costs O(n²), done `n` times for O(n³) in total, which does not change the overall bound.
- Time is the same in the best, average and worst case, regardless of how many edges the graph has.

### Space complexity: O(n²)
- The input matrix and the `prev` / `curr` matrices each hold `n x n` values.
- At any moment only a constant number (about 2–3) of such matrices are alive, so auxiliary space is **O(n²)**.
- Note: this version allocates a new matrix on every outer iteration (n allocations of size n² in total). Old ones are garbage-collected, but an **in-place** update would avoid this overhead and use a single matrix.

### Summary
| Metric | Complexity |
|---|---|
| Time | O(n³) |
| Space | O(n²) |

### When to use it
Floyd–Warshall is ideal for **dense graphs** and for small to medium `n` where all-pairs distances are needed. It handles negative edge weights (but not negative cycles). For large sparse graphs, running Dijkstra from every vertex, at about O(V(E + V log V)), is usually faster.